In [511]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
from scipy.stats import chi2_contingency, ttest_ind
from IPython.display import display, HTML

## Common Functions to Fetch Annotator Results, i.e. Events and Labels for Benchmark Scenarios

In [512]:
#  takes in an annotator output and returns a dictionary with the events as keys and the binary labellings as values

def parse_events_with_labels(file_path):
    """
    Parse a JSON file containing event nodes and extract their C/I/K polarity labels.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        dict: Mapping of event labels to their C/I/K polarity strings
              e.g., {"Historical buildings are demolished": "C+I+K+", ...}
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Parse each line as a separate JSON object
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]
    
    # Find the being node (first node with kind "being")
    being_node = None
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'being':
            being_node = node_obj
            break
    
    if not being_node:
        return {}
                    
    
    # Create a mapping from event labels to their C/I/K values
    event_labels = {}
    
    for link in being_node.get('links', []):
        to_node_label = link.get('to_node')
        b_link_value = link.get('link', {}).get('value')
        
        if to_node_label and b_link_value:
            event_labels[to_node_label] = [b_link_value]
    
    # Filter to only include actual events
    event_nodes = {
        node_obj['node']['label'] 
        for node_obj in nodes 
        if node_obj.get('node', {}).get('kind') == 'event'
    }

    # attach the utility value to the event labels
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'event':
            event_label = node_obj['node']['label']
            for link in node_obj.get('links', []):
                if link.get('to_node') == being_node.get('node', {}).get('label') and link.get('link', {}).get('kind') == 'utility':
                    utility_value = link.get('link', {}).get('value')
                    event_labels[event_label].append(utility_value)

    
    return {label: value for label, value in event_labels.items() if label in event_nodes}

In [513]:
def annotator_df_maker(annotator_output_path, annotator_input_path):
    """
    Create a DataFrame from the annotated output and input files for a given annotator.
    
    Args:
        annotator_output_path: Path to the annotated output JSON file
        annotator_input_path: Path to the annotated input JSON file
    """
    annotator_megadf = []
    for folder in annotator_output_path.iterdir():
        if folder.is_dir():
            # print(f"Processing folder: {folder.name}")
            evitability = "Inevitable" if "inevitable" in folder.name else "Evitable"
            means_side_effect = "CC (Means)" if "cc" in folder.name else "COC (SideEff)"
            co_omission = "Commission" if "action_yes" in folder.name else "Omission"

        for json_file in folder.glob("*choice_1.json"):
            # print(f"Processing file: {json_file} in folder: {folder.name}")
            sid = json_file.stem.split("_")[0]  # Extract ScenarioID from filename
            # print(f"Processing ScenarioID: {sid}")

            # find the json file in the input directory with the same name as the folder name in the output directory
            input_json_file = annotator_input_path / f'{folder.name}.json'
            # print if the input json file exists
            # if input_json_file.exists():
                # print(f"Found input JSON file: {input_json_file}")
            # pick out the individual json block with the "id" value that matches the sid value from the output directory
            with open(input_json_file, 'r') as f:
                input_data = json.load(f)
                for k in input_data:
                    # print(f"Checking block with id: {k['id']} against ScenarioID: {sid}")
                    if str(k['id']) == sid:
                        # print(f"Found matching block for ScenarioID: {sid} in input file: {input_json_file}")
                        # extract the "text" sceanrio and the "options" list from the block
                        scenario_text = k['text']
                        options = k['options']
                        break
            
            event_labels = parse_events_with_labels(json_file)
            
            for event, cik_value in event_labels.items():
                c_value = cik_value[0][1]  # C polarity
                i_value = cik_value[0][3]  # I polarity
                k_value = cik_value[0][5]  # K polarity
                event_utility = cik_value[1]  # Utility value
                
                annotator_megadf.append({
                    "scenario_id": sid,
                    "folder_name": folder.name,
                    "evitability": evitability,
                    "means_side_effect": means_side_effect,
                    "commission_omission": co_omission,
                    "scenario_text": scenario_text,
                    "options": options,
                    "event": event,
                    "c": c_value,
                    "i": i_value,
                    "k": k_value,
                    "utility": event_utility  # Utility value
                })
                # print(f"Added event '{event}' with C={c_value}, I={i_value}, K={k_value}, utility={event_utility} to the DataFrame for ScenarioID: {sid}")

    folder_order = [
    "cc_evitable_action_yes_stories",
    "cc_evitable_prevention_no_stories",
    "cc_inevitable_action_yes_stories",
    "cc_inevitable_prevention_no_stories",
    "coc_evitable_action_yes_stories",
    "coc_evitable_prevention_no_stories",
    "coc_inevitable_action_yes_stories",
    "coc_inevitable_prevention_no_stories"
    ]   

    
    # annotator_megadf.sort(key=lambda x: (folder_order.index(x["folder_name"]), int(x["scenario_id"])))

    # reorder the rows by folder name only
    annotator_megadf.sort(key=lambda x: folder_order.index(x["folder_name"]))
    # then reorder the rows by scenario_id within each folder name
    annotator_megadf.sort(key=lambda x: int(x["scenario_id"]))
    # convert to a DataFrame
    annotator_megadf = pd.DataFrame(annotator_megadf)
    # preview the DataFrame
    print(f"Annotator DataFrame for {annotator_output_path.name}:")
    # display(HTML(annotator_megadf[0:10].to_html()))
    return annotator_megadf

## Starting Point A -- Fetch All Our Annotator Outputs Into A Big Dataframe

### One DF for mild scenarios, one DF for severe scenarios --- we will create both but only use the mild annotations one for now because Exp2 only uses mild inputs

In [514]:
mild_annotated_output_path = Path().resolve() / "../../annotated_outputs/franken/conditions_mild_harm_mild_good"
severe_annotated_output_path = Path().resolve() / "../../annotated_outputs/franken/conditions_severe_harm_very_good"

mild_annotated_input_path = Path().resolve() / "../../scenarios_inputs/franken/conditions_mild_harm_mild_good"
severe_annotated_input_path = Path().resolve() / "../../scenarios_inputs/franken/conditions_severe_harm_very_good"

In [515]:
# Create a mild scenario annotator DataFrame
mild_annotator_megadf = annotator_df_maker(mild_annotated_output_path, mild_annotated_input_path)

# Create a severe scenario annotator DataFrame
severe_annotator_megadf = annotator_df_maker(severe_annotated_output_path, severe_annotated_input_path)

Annotator DataFrame for conditions_mild_harm_mild_good:
Annotator DataFrame for conditions_severe_harm_very_good:


In [516]:
# preview the mild annotator DataFrame
print("Mild Annotator DataFrame:")
display(HTML(mild_annotator_megadf[0:10].to_html()))
print("Number of rows in mild annotator DataFrame:", len(mild_annotator_megadf))

Mild Annotator DataFrame:


,scenario_id,folder_name,evitability,means_side_effect,commission_omission,scenario_text,options,event,c,i,k,utility
0,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",I initiate renovation work in the park,+,+,+,20
1,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",The community temporarily loses access to the recreational space,+,+,+,-10
2,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",Residents experience inconvenience due to park closure,+,+,+,-20
3,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",Residents feel disappointment over the temporary loss of the park,+,+,+,-20
4,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",The park's facilities are improved after renovation,+,+,+,80
5,0,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,"I, an Architect, face a moral dilemma. My colleague has the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I prevent my colleague's park renovation, there would not be the temporary deprivation of the recreational space and the resulting inconvenience and disappointment.","{'1': 'prevent my colleague's park renovation.', '2': 'not prevent my colleague's park renovation.'}",The community r

Number of rows in mild annotator DataFrame: 1198


## Starting Point B --- Fetch Human Exp2 Data Into A DataFrame

### Exp2 DF 1 : Raw Exp 2 Input Items (Intermediate preprocessing step)

In [517]:
exp2_data_path = Path().resolve() / "../../human_data/franken/exp2_moralperm-intent-rating"
exp2_raw_df = []
# go through all json files in the exp2_data_path
for json_file in exp2_data_path.glob("*.json"):
    # print(f"Processing file: {json_file.name}")
    with open(json_file, 'r') as f:
        curr_batch_list = json.load(f)
        for exp2_item in curr_batch_list:
            exp2_raw_df.append(exp2_item)
# reorder the exp2_raw_df by the "folder_name" key according to the folder_order list
condition_order = [
    "cc_evitable_action_yes",
    "cc_evitable_prevention_no",
    "cc_inevitable_action_yes",
    "cc_inevitable_prevention_no",
    "coc_evitable_action_yes",
    "coc_evitable_prevention_no",
    "coc_inevitable_action_yes",
    "coc_inevitable_prevention_no"
]
exp2_raw_df.sort(key=lambda x: condition_order.index(x["condition"]))
# reorder the exp2_raw_df by the "scenario_id" key in ascending order
exp2_raw_df.sort(key=lambda x: int(x["scenario_id"]))
exp2_raw_df = pd.DataFrame(exp2_raw_df)
# print("Raw DataFrame from Experiment 2:")
# display(HTML(exp2_raw_df[0:10].to_html()))
# print(f"Total number of rows in Exp2 raw DataFrame: {len(exp2_raw_df)}")

### Exp 2 DF 2 : Coded Input Items w/ Human Ratings (Intermediate pre-processing step)

In [518]:
human_ratings = pd.read_csv(Path().resolve() / "../../human_data/franken/exp2_moralperm-intent-rating/data_long_format.csv")
human_ratings = human_ratings.drop(columns=['scenario_harm', 'split'])
# count number of unique combinations of scenario_id + causal_structure + evitability + action
exp2_ratings_df = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action']).size().reset_index(name='counts') # each scenario got rated by ~20-25 participants
# add a column of average rating of moral permissibility and intention for each unique combination
avg_ratings = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action'])[['permissibility_rating', 'intention_rating']].mean().reset_index()
exp2_ratings_df = exp2_ratings_df.merge(avg_ratings, on=['scenario_id', 'causal_structure', 'evitability', 'action'])
exp2_ratings_df = exp2_ratings_df.rename(columns={
    'permissibility_rating': 'avg_permissibility_rating',
    'intention_rating': 'avg_intention_rating'
})
print("\n Explainer: \n causal_structure (0 for means (cc), 1 for side effect (coc)) \n action (0 for commission i.e. action_yes, 1 for omission i.e. prevention_no) \n evitability (0 for evitable, 1 for inevitable) \n avg_permissibility_rating (average moral permissibility rating for that scenario) \n avg_intention_rating (average intention rating for that scenario)")
# display(HTML(exp2_ratings_df[:10].to_html()))
# print(f"Total number of scenarios rated in Exp2: {len(exp2_ratings_df)}")


 Explainer: 
 causal_structure (0 for means (cc), 1 for side effect (coc)) 
 action (0 for commission i.e. action_yes, 1 for omission i.e. prevention_no) 
 evitability (0 for evitable, 1 for inevitable) 
 avg_permissibility_rating (average moral permissibility rating for that scenario) 
 avg_intention_rating (average intention rating for that scenario)


### FINAL Exp2 Dataframe = DF 1 + DF 2 

In [519]:
# Simply tack on the avg_intention_rating and avg_permissibility_rating columns to the exp2_raw_df dataframe, no need to do any matching since they are both in the same order and have the same number of rows
exp2_inputs_df = exp2_raw_df.copy()
exp2_inputs_df['avg_permissibility_rating'] = exp2_ratings_df['avg_permissibility_rating']
exp2_inputs_df['avg_intention_rating'] = exp2_ratings_df['avg_intention_rating']
print("Final Experiment 2 DataFrame with average human ratings added (please scroll horizontally):")
display(HTML(exp2_inputs_df[0:10].to_html()))
print(f"Total number of rows in final Exp2 DataFrame: {len(exp2_inputs_df)}")

Final Experiment 2 DataFrame with average human ratings added (please scroll horizontally):


,permissibility_question,intention_question,context,opportunity,structure_sentence,evitability_sentence,action_sentence,scenario_id,condition,avg_permissibility_rating,avg_intention_rating
0,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya has the opportunity to renovate a park in the city.,"As a means to improve some of the park's facilitie, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents.","If Maya does not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.",Maya renovates the park.,0,cc_evitable_action_yes,4.153846,1.923077
1,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya's colleague has the opportunity to renovate a park in the city.,"As a means to improve some of the park's facilitie, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents.","If Maya prevents her colleague's park renovation, there would not be the temporary deprivation of the recreational space and the resulting inconvenience and disappointment.",Maya does not prevent her colleague's park renovation.,0,cc_evitable_prevention_no,4.280000,1.960000
2,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya has the opportunity to renovate a park in the city.,"As a means to improve some of the park's facilitie, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents.","If Maya does not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would still happen because of a city mandate for construction.",Maya renovates the park.,0,cc_inevitable_action_yes,3.952381,2.476190
3,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya's colleague has the opportunity to renovate a park in the city.,"As a means to improve some of the park's facilitie, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents.","If Maya prevents her colleague's park renovation, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would still happen because of a city mandate for construction.",Maya does not prevent her colleague's park renovation.,0,cc_inevitable_prevention_no,3.952381,2.666667
4,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya has the opportunity to implement a new bike lane system throughout the city.,"The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city.","If Maya does not implement the bike lane system, the increase in parking prices would not happen.",Maya implements the new bike lane system.,0,coc_evitable_action_yes,4.272727,2.045455
5,What Maya did is morally permissible.,Maya intended for the negative outcome to occur.,"Maya, an Architect, faces a moral dilemma.",Maya's colleague has the opportunity to implement a new bike lane system throughout the city.,"The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city.","If Maya prevents her colleague's implementation of the bik

Total number of rows in final Exp2 DataFrame: 80


## Next Stage: Matching Up Annotator Scenarios with Exp2 Ratings

#### - Our Prepared Annotator inputs DF = `mild_annotator_megadf`
#### - Our Prepared Exp2 inputs DF = `exp2_inputs_df`

Both contain data for the same 80 scenarios. The row count in Exp2 DF is 80.

We can cleanly match Annotator DF with Exp2 DF; there is a 1:1 matching between annotator input scenarios and Exp2 inputs. 

BUT each annotator input scenarios gets multiple generated events (like 5-10). So the row count in Annotator DF is much higher than 80.

Still, each set of generated events gets the same human intentionality/permissibility ratings copy-pasted across them, because they belong to the same scenario.

In [520]:
# loop through the mild_annotator_megadf and at each row, find the matching folder_name AND scenario_id in the exp2_inputs_df and add the avg_permissibility_rating and avg_intention_rating to the mild_annotator_megadf for that row. Multiple rows in the mild_annotator_megadf will match to the same scenario_id and folder_name in the exp2_inputs_df, so they will get the same avg_permissibility_rating and avg_intention_rating values. 
for i, row in mild_annotator_megadf.iterrows():
    folder_name = row['folder_name']
    folder_name = folder_name.replace("_stories", "")
    # print(f"Processing row {i} with folder_name: {folder_name} and scenario_id: {row['scenario_id']}")
    scenario_id = row['scenario_id']
    matching_row = exp2_inputs_df[(exp2_inputs_df['condition'] == folder_name) & (exp2_inputs_df['scenario_id'].astype(str) == str(scenario_id))]
    if not matching_row.empty:
        # print(f"Found matching row in exp2_inputs_df for row {i} in mild_annotator_megadf with folder_name: {folder_name} and scenario_id: {scenario_id}")
        mild_annotator_megadf.at[i, 'avg_permissibility_rating'] = matching_row['avg_permissibility_rating'].values[0]
        mild_annotator_megadf.at[i, 'avg_intention_rating'] = matching_row['avg_intention_rating'].values[0]

# print the mild_annotator_megadf with the new columns added
print("Mild Annotator DataFrame with Exp2's average human ratings added (please scroll horizontally):")
display(HTML(mild_annotator_megadf[0:10].to_html()))

Mild Annotator DataFrame with Exp2's average human ratings added (please scroll horizontally):


,scenario_id,folder_name,evitability,means_side_effect,commission_omission,scenario_text,options,event,c,i,k,utility,avg_permissibility_rating,avg_intention_rating
0,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",I initiate renovation work in the park,+,+,+,20,4.153846,1.923077
1,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",The community temporarily loses access to the recreational space,+,+,+,-10,4.153846,1.923077
2,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",Residents experience inconvenience due to park closure,+,+,+,-20,4.153846,1.923077
3,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",Residents feel disappointment over the temporary loss of the park,+,+,+,-20,4.153846,1.923077
4,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","{'1': 'renovate the park.', '2': 'not renovate the park.'}",The park's facilities are improved after renovation,+,+,+,80,4.153846,1.923077
5,0,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,"I, an Architect, face a moral dilemma. My colleague has the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I prevent my colleague's park renovation, there would not be the temporary deprivation of the recreational space and the resulting inconvenience an

#### This is the fully-processed dataframe for Exp2 analysis. `mild_annotator_megadf` (above) now contains all the (mild) annotator input scenarios, the set of generated events for each scenario, each event's C/I/K and utility ratings generated by the annotator, AND the Exp2 intentionality/permissibility human ratings that were provided for that scenario!

#### **Final Stage of Exp2 analysis:** Now, we need to isolate to a single event per scenario. Right now, you will see 5-10 consecutive rows with the SAME human ratings; this is because they are all events belonging to the same scenario. It needs to go from 5-10 events to 1 event.

#### We will do this by picking out the primary harm event for each scenario out of `mild_annotator_megadf`.

In [ ]:
#TODO: this cell is copied from the old notebook; need to see if I can simply use the same row numbers or map them to the correct per-scenario primary harm event rows in the new mild_annotator_megadf above

primary_harm_rows_exp2 = [2, 9, 12, 21, 25, 34, 39, 49, 53, 61,                # primary harm for scenarios 1-10
                          64, 69, 76, 78, 85, 91, 93, 100, 103, 107,           # primary harm for scenarios 11-20
                          109, 113, 117, 123, 126, 131, 135, 141, 144, 148]    # primary harm for scenarios 21-30